In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import array_bounds
from rasterio.warp import transform_bounds
from pyproj import Transformer
from rasterio.windows import Window, from_bounds


tif_path = Path("Output/GEDI/GEDI03_rh100_mean_2019108_2025190_002_05.tiff")
# Bordeaux with extra westward margin so the Atlantic is visible
bordeaux_lon, bordeaux_lat = -0.5792, 44.8378

with rasterio.open(tif_path) as src:
    transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    x, y = transformer.transform(bordeaux_lon, bordeaux_lat)

    if src.crs.is_geographic:
        left = x - 1.5
        right = x + 1.5
        bottom = y - 1.0
        top = y + 1.0
    else:
        west, south = transformer.transform(bordeaux_lon - 1.5, bordeaux_lat - 1.0)
        east, north = transformer.transform(bordeaux_lon + 1.5, bordeaux_lat + 1.0)
        left = min(west, east)
        right = max(west, east)
        bottom = min(south, north)
        top = max(south, north)

    window = from_bounds(left, bottom, right, top, transform=src.transform)
    window = window.intersection(Window(0, 0, src.width, src.height))
    zoom = src.read(1, window=window)
    zoom_transform = src.window_transform(window)
    zoom_left, zoom_bottom, zoom_right, zoom_top = transform_bounds(
        src.crs,
        "EPSG:4326",
        *array_bounds(zoom.shape[0], zoom.shape[1], zoom_transform),
    )

    figure, map_ax = plt.subplots(
        figsize=(10, 10),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True,
    )
    raster = map_ax.imshow(
        zoom,
        extent=(zoom_left, zoom_right, zoom_bottom, zoom_top),
        origin="upper",
        cmap="Greens",
        vmin=0,
        vmax=45,
        transform=ccrs.PlateCarree(),
    )
    map_ax.scatter(
        [bordeaux_lon],
        [bordeaux_lat],
        c="red",
        s=50,
        edgecolors="white",
        linewidths=1.2,
        transform=ccrs.PlateCarree(),
        zorder=3,
    )
    map_ax.text(
        bordeaux_lon + 0.02,
        bordeaux_lat + 0.02,
        "Bordeaux",
        fontsize=14,
        color="black",
        ha="left",
        va="bottom",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.7, "pad": 2},
        transform=ccrs.PlateCarree(),
        zorder=4,
    )
    map_ax.set_xlabel("Longitude", fontsize=14)
    map_ax.set_ylabel("Latitude", fontsize=14)
    gridlines = map_ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        alpha=0.5,
        linestyle="--",
    )
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.xlabel_style = {"size": 12}
    gridlines.ylabel_style = {"size": 12}

    # Colorbar axes in map-axes coordinates, so it matches the map height exactly
    colorbar_ax = map_ax.inset_axes([1.03, 0, 0.04, 1])
    colorbar = figure.colorbar(raster, cax=colorbar_ax, orientation="vertical")
    colorbar.set_label("Boshoogte (m)", fontsize=20)
    colorbar.ax.tick_params(labelsize=18)
    Path("Output/figures").mkdir(exist_ok=True)
    figure.savefig(
        "Output/figures/vegetatiehoogte_bordeaux.svg",
        format="svg",
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
from IPython.display import display

# Child-friendly version of the figure above: no numbers on the colorbar,
# just "Laag" at the bottom, "Hoog" at the top and an arrow from low to high.
low_label, high_label = "Laag", "Hoog"

# Remove earlier additions so re-running this cell does not stack them
for artist in [text for text in colorbar.ax.texts if text.get_gid() == "kids_colorbar"]:
    artist.remove()

colorbar.set_ticks([])
colorbar.ax.text(0.5, -0.01, low_label, transform=colorbar.ax.transAxes,
                 ha="center", va="top", fontsize=22, gid="kids_colorbar")
colorbar.ax.text(0.5, 1.01, high_label, transform=colorbar.ax.transAxes,
                 ha="center", va="bottom", fontsize=22, gid="kids_colorbar")
colorbar.ax.annotate(
    "",
    xy=(2.0, 0.97),
    xytext=(2.0, 0.03),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "-|>", "linewidth": 3, "color": "black", "mutation_scale": 30},
    annotation_clip=False,
    gid="kids_colorbar",
)
colorbar.ax.yaxis.labelpad = 45  # keep the colorbar title clear of the arrow

kids_svg = Path("Output/figures/vegetatiehoogte_bordeaux_kids.svg")
figure.savefig(kids_svg, format="svg", bbox_inches="tight")
display(figure)
print(f"Saved {kids_svg}")